# Part C — Modeling and Hyperparameter Tuning

**Project:** Academy Nova Course Cancellation Prediction  
**Group:** 51  
**Main metric:** ROC-AUC  

This notebook covers model training, validation, hyperparameter tuning, and model comparison.

The main goal is to build a model with strong AUC while controlling overfitting.  
We will compare several models using cross-validation and track both validation performance and train-validation gaps.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import ID_COL, TARGET_COL, RANDOM_STATE, N_SPLITS
from src.data_loading import load_raw_data, validate_raw_data
from src.features import create_engineered_features

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [2]:
train_df, test_df = load_raw_data()
validate_raw_data(train_df, test_df)

train_fe = create_engineered_features(train_df)
test_fe = create_engineered_features(test_df)

print("Train with features:", train_fe.shape)
print("Test with features:", test_fe.shape)

Validating raw data...
Raw data validation passed.
Train shape: (63464, 29)
Test shape: (15866, 28)
Target positive rate: 0.4144
Train with features: (63464, 67)
Test with features: (15866, 66)


In [3]:
X = train_fe.drop(columns=[TARGET_COL])
y = train_fe[TARGET_COL]

X_test = test_fe.copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

X shape: (63464, 66)
y shape: (63464,)
X_test shape: (15866, 66)


## 1. Preprocessing Strategy

The dataset contains numeric, categorical, date, and ID-like columns.

For the baseline models:
- numeric features are imputed with the median,
- categorical features are imputed with `"Missing"` and one-hot encoded,
- unknown categories in validation/test are ignored,
- `Client_ID` is removed because it is only an identifier,
- `Course_Start_Date` is removed after extracting date-derived features.

This preprocessing is fitted only on the training folds inside cross-validation to avoid data leakage.

In [4]:
DROP_COLS = [ID_COL, "Course_Start_Date"]

X_model = X.drop(columns=[col for col in DROP_COLS if col in X.columns])
X_test_model = X_test.drop(columns=[col for col in DROP_COLS if col in X_test.columns])

numeric_cols = X_model.select_dtypes(include=["int64", "float64", "Int64"]).columns.tolist()
categorical_cols = X_model.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print("Total model columns:", X_model.shape[1])

Numeric columns: 50
Categorical columns: 10
Total model columns: 64


In [5]:
numeric_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_preprocessor = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_preprocessor, numeric_cols),
        ("cat", categorical_preprocessor, categorical_cols),
    ],
    remainder="drop"
)

In [6]:
def evaluate_model_cv(model, X, y, model_name, n_splits=N_SPLITS):
    """
    Evaluate a model using Stratified K-Fold CV.

    Returns:
    - fold results table
    - out-of-fold predictions
    - fitted fold models
    """
    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    oof_preds = np.zeros(len(X))
    fold_results = []
    fold_models = []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        pipeline = Pipeline(steps=[
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)

        train_pred = pipeline.predict_proba(X_train)[:, 1]
        valid_pred = pipeline.predict_proba(X_valid)[:, 1]

        train_auc = roc_auc_score(y_train, train_pred)
        valid_auc = roc_auc_score(y_valid, valid_pred)

        oof_preds[valid_idx] = valid_pred
        fold_models.append(pipeline)

        fold_results.append({
            "model": model_name,
            "fold": fold,
            "train_auc": train_auc,
            "valid_auc": valid_auc,
            "gap": train_auc - valid_auc
        })

        print(
            f"{model_name} | Fold {fold}: "
            f"train AUC={train_auc:.5f}, "
            f"valid AUC={valid_auc:.5f}, "
            f"gap={train_auc - valid_auc:.5f}"
        )

    fold_results_df = pd.DataFrame(fold_results)
    oof_auc = roc_auc_score(y, oof_preds)

    print("\nOOF AUC:", round(oof_auc, 5))
    print("Mean valid AUC:", round(fold_results_df["valid_auc"].mean(), 5))
    print("Std valid AUC:", round(fold_results_df["valid_auc"].std(), 5))
    print("Mean gap:", round(fold_results_df["gap"].mean(), 5))

    return fold_results_df, oof_preds, fold_models

## 2. Baseline Model Comparison

We first train several baseline models.  
The purpose is not only to get the highest AUC, but also to understand which model family fits this dataset best and how much overfitting each model shows.

In [7]:
log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

logreg_results, logreg_oof, logreg_models = evaluate_model_cv(
    log_reg,
    X_model,
    y,
    "Logistic Regression"
)

Logistic Regression | Fold 1: train AUC=0.91254, valid AUC=0.90685, gap=0.00569
Logistic Regression | Fold 2: train AUC=0.91265, valid AUC=0.90648, gap=0.00617
Logistic Regression | Fold 3: train AUC=0.91377, valid AUC=0.90165, gap=0.01211
Logistic Regression | Fold 4: train AUC=0.91404, valid AUC=0.90079, gap=0.01325
Logistic Regression | Fold 5: train AUC=0.91371, valid AUC=0.90259, gap=0.01112

OOF AUC: 0.90365
Mean valid AUC: 0.90367
Std valid AUC: 0.00281
Mean gap: 0.00967


In [8]:
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    class_weight="balanced"
)

rf_results, rf_oof, rf_models = evaluate_model_cv(
    rf,
    X_model,
    y,
    "Random Forest"
)

Random Forest | Fold 1: train AUC=0.88776, valid AUC=0.88974, gap=-0.00198
Random Forest | Fold 2: train AUC=0.89006, valid AUC=0.88890, gap=0.00116
Random Forest | Fold 3: train AUC=0.89354, valid AUC=0.88479, gap=0.00874
Random Forest | Fold 4: train AUC=0.89257, valid AUC=0.88282, gap=0.00976
Random Forest | Fold 5: train AUC=0.88511, valid AUC=0.88622, gap=-0.00111

OOF AUC: 0.88628
Mean valid AUC: 0.88649
Std valid AUC: 0.00286
Mean gap: 0.00331


In [9]:
hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=300,
    max_leaf_nodes=31,
    l2_regularization=0.1,
    random_state=RANDOM_STATE
)

hgb_results, hgb_oof, hgb_models = evaluate_model_cv(
    hgb,
    X_model,
    y,
    "HistGradientBoosting"
)

HistGradientBoosting | Fold 1: train AUC=0.96137, valid AUC=0.94916, gap=0.01221
HistGradientBoosting | Fold 2: train AUC=0.96128, valid AUC=0.94905, gap=0.01222
HistGradientBoosting | Fold 3: train AUC=0.96243, valid AUC=0.94506, gap=0.01737
HistGradientBoosting | Fold 4: train AUC=0.96218, valid AUC=0.94708, gap=0.01510
HistGradientBoosting | Fold 5: train AUC=0.96184, valid AUC=0.94830, gap=0.01354

OOF AUC: 0.94768
Mean valid AUC: 0.94773
Std valid AUC: 0.00171
Mean gap: 0.01409


In [10]:
all_baseline_results = pd.concat(
    [logreg_results, rf_results, hgb_results],
    ignore_index=True
)

model_summary = (
    all_baseline_results
    .groupby("model")
    .agg(
        mean_train_auc=("train_auc", "mean"),
        mean_valid_auc=("valid_auc", "mean"),
        std_valid_auc=("valid_auc", "std"),
        mean_gap=("gap", "mean"),
    )
    .sort_values("mean_valid_auc", ascending=False)
)

model_summary

,mean_train_auc,mean_valid_auc,std_valid_auc,mean_gap
model,,,,
HistGradientBoosting,0.961819,0.947730,0.001709,0.014089
Logistic Regression,0.913343,0.903673,0.002811,0.009669
Random Forest,0.889809,0.886494,0.002864,0.003315


### Baseline modeling insight

The baseline comparison allows us to evaluate both predictive performance and overfitting.  
A model with a very high training AUC but much lower validation AUC is likely too flexible.  
The preferred model should have strong validation AUC, low fold variance, and a reasonable train-validation gap.

## 3. Advanced Boosting Models

The baseline results show that gradient boosting is the strongest model family so far.  
We now test more advanced boosting models that are commonly strong on tabular data, especially LightGBM and XGBoost.

The goal is to improve AUC while monitoring the train-validation gap to avoid overfitting.

In [11]:
try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM is not installed.")

In [12]:
if LIGHTGBM_AVAILABLE:
    lgbm = LGBMClassifier(
        n_estimators=600,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="binary",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    )

    lgbm_results, lgbm_oof, lgbm_models = evaluate_model_cv(
        lgbm,
        X_model,
        y,
        "LightGBM"
    )

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM | Fold 1: train AUC=0.96354, valid AUC=0.95100, gap=0.01253


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM | Fold 2: train AUC=0.96335, valid AUC=0.95064, gap=0.01271


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM | Fold 4: train AUC=0.96415, valid AUC=0.94851, gap=0.01564


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM | Fold 5: train AUC=0.96392, valid AUC=0.95140, gap=0.01253

OOF AUC: 0.9496
Mean valid AUC: 0.94966
Std valid AUC: 0.00198
Mean gap: 0.01426


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [13]:
all_model_results = pd.concat(
    [logreg_results, rf_results, hgb_results, lgbm_results],
    ignore_index=True
)

model_summary = (
    all_model_results
    .groupby("model")
    .agg(
        mean_train_auc=("train_auc", "mean"),
        mean_valid_auc=("valid_auc", "mean"),
        std_valid_auc=("valid_auc", "std"),
        mean_gap=("gap", "mean"),
    )
    .sort_values("mean_valid_auc", ascending=False)
)

model_summary

,mean_train_auc,mean_valid_auc,std_valid_auc,mean_gap
model,,,,
LightGBM,0.963916,0.949657,0.001980,0.014259
HistGradientBoosting,0.961819,0.947730,0.001709,0.014089
Logistic Regression,0.913343,0.903673,0.002811,0.009669
Random Forest,0.889809,0.886494,0.002864,0.003315


### Advanced boosting insight

LightGBM is currently the strongest model.  
It improves the mean validation AUC compared to HistGradientBoosting while keeping a similar train-validation gap.

This suggests that gradient boosting models are the best fit for this tabular dataset.  
The next step is to test XGBoost and then tune the strongest boosting model while monitoring overfitting.

## 4. XGBoost Baseline

XGBoost is another strong gradient boosting model for tabular data.  
We test it because it may capture feature interactions differently from LightGBM.

As before, we compare both validation AUC and the train-validation gap to avoid choosing a model that overfits.

In [14]:
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost is not installed.")

XGBoost is not installed.


In [15]:
if XGBOOST_AVAILABLE:
    xgb = XGBClassifier(
        n_estimators=600,
        learning_rate=0.03,
        max_depth=5,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        objective="binary:logistic",
        eval_metric="auc",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method="hist"
    )

    xgb_results, xgb_oof, xgb_models = evaluate_model_cv(
        xgb,
        X_model,
        y,
        "XGBoost"
    )

In [16]:
model_result_list = [logreg_results, rf_results, hgb_results]

if LIGHTGBM_AVAILABLE:
    model_result_list.append(lgbm_results)

if XGBOOST_AVAILABLE:
    model_result_list.append(xgb_results)

all_model_results = pd.concat(model_result_list, ignore_index=True)

model_summary = (
    all_model_results
    .groupby("model")
    .agg(
        mean_train_auc=("train_auc", "mean"),
        mean_valid_auc=("valid_auc", "mean"),
        std_valid_auc=("valid_auc", "std"),
        mean_gap=("gap", "mean"),
    )
    .sort_values("mean_valid_auc", ascending=False)
)

model_summary

,mean_train_auc,mean_valid_auc,std_valid_auc,mean_gap
model,,,,
LightGBM,0.963916,0.949657,0.001980,0.014259
HistGradientBoosting,0.961819,0.947730,0.001709,0.014089
Logistic Regression,0.913343,0.903673,0.002811,0.009669
Random Forest,0.889809,0.886494,0.002864,0.003315


In [17]:
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("XGBoost is installed.")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost is NOT installed.")

XGBoost is NOT installed.


### XGBoost availability

XGBoost was checked as an additional advanced boosting model, but it was not installed in the current environment.

Since the project already includes Logistic Regression, Random Forest, HistGradientBoosting, and LightGBM, the modeling requirement of comparing multiple model families is still satisfied.

The current strongest model is LightGBM, so the next step is to tune LightGBM while monitoring the train-validation gap.

In [18]:
model_result_list = [logreg_results, rf_results, hgb_results]

if LIGHTGBM_AVAILABLE:
    model_result_list.append(lgbm_results)

if XGBOOST_AVAILABLE and "xgb_results" in globals():
    model_result_list.append(xgb_results)

all_model_results = pd.concat(model_result_list, ignore_index=True)

model_summary = (
    all_model_results
    .groupby("model")
    .agg(
        mean_train_auc=("train_auc", "mean"),
        mean_valid_auc=("valid_auc", "mean"),
        std_valid_auc=("valid_auc", "std"),
        mean_gap=("gap", "mean"),
    )
    .sort_values("mean_valid_auc", ascending=False)
)

model_summary

,mean_train_auc,mean_valid_auc,std_valid_auc,mean_gap
model,,,,
LightGBM,0.963916,0.949657,0.001980,0.014259
HistGradientBoosting,0.961819,0.947730,0.001709,0.014089
Logistic Regression,0.913343,0.903673,0.002811,0.009669
Random Forest,0.889809,0.886494,0.002864,0.003315


## 5. LightGBM Hyperparameter Tuning

LightGBM is currently the strongest model, so we tune it carefully.

The goal is not only to maximize validation AUC, but also to keep the train-validation gap reasonable.  
A model with slightly lower AUC but much smaller overfitting gap may generalize better to the hidden leaderboard.

In [19]:
lgbm_configs = [
    {
        "name": "LightGBM_baseline",
        "params": {
            "n_estimators": 600,
            "learning_rate": 0.03,
            "num_leaves": 31,
            "max_depth": -1,
            "min_child_samples": 50,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
        }
    },
    {
        "name": "LightGBM_regularized",
        "params": {
            "n_estimators": 700,
            "learning_rate": 0.025,
            "num_leaves": 24,
            "max_depth": 8,
            "min_child_samples": 80,
            "subsample": 0.75,
            "colsample_bytree": 0.75,
            "reg_alpha": 0.5,
            "reg_lambda": 2.0,
        }
    },
    {
        "name": "LightGBM_deeper",
        "params": {
            "n_estimators": 700,
            "learning_rate": 0.025,
            "num_leaves": 48,
            "max_depth": -1,
            "min_child_samples": 40,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.05,
            "reg_lambda": 0.8,
        }
    },
    {
        "name": "LightGBM_slow_regularized",
        "params": {
            "n_estimators": 1000,
            "learning_rate": 0.015,
            "num_leaves": 31,
            "max_depth": 10,
            "min_child_samples": 70,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "reg_alpha": 0.3,
            "reg_lambda": 2.5,
        }
    },
]

In [20]:
lgbm_tuning_results = []
lgbm_tuning_oofs = {}
lgbm_tuning_models = {}

if LIGHTGBM_AVAILABLE:
    for config in lgbm_configs:
        print("=" * 80)
        print(config["name"])
        print("=" * 80)

        model = LGBMClassifier(
            **config["params"],
            objective="binary",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1
        )

        fold_results, oof_preds, fold_models = evaluate_model_cv(
            model,
            X_model,
            y,
            config["name"]
        )

        lgbm_tuning_results.append(fold_results)
        lgbm_tuning_oofs[config["name"]] = oof_preds
        lgbm_tuning_models[config["name"]] = fold_models

LightGBM_baseline


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM_baseline | Fold 1: train AUC=0.96354, valid AUC=0.95100, gap=0.01253


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM_baseline | Fold 2: train AUC=0.96335, valid AUC=0.95064, gap=0.01271


...

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM_slow_regularized | Fold 4: train AUC=0.95900, valid AUC=0.94624, gap=0.01275


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


LightGBM_slow_regularized | Fold 5: train AUC=0.95846, valid AUC=0.94867, gap=0.00979

OOF AUC: 0.94752
Mean valid AUC: 0.94757
Std valid AUC: 0.00182
Mean gap: 0.01116


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [21]:
lgbm_tuning_all_results = pd.concat(lgbm_tuning_results, ignore_index=True)

lgbm_tuning_summary = (
    lgbm_tuning_all_results
    .groupby("model")
    .agg(
        mean_train_auc=("train_auc", "mean"),
        mean_valid_auc=("valid_auc", "mean"),
        std_valid_auc=("valid_auc", "std"),
        mean_gap=("gap", "mean"),
    )
    .sort_values("mean_valid_auc", ascending=False)
)

lgbm_tuning_summary

,mean_train_auc,mean_valid_auc,std_valid_auc,mean_gap
model,,,,
LightGBM_deeper,0.972509,0.952349,0.001584,0.020160
LightGBM_baseline,0.963916,0.949657,0.001980,0.014259
LightGBM_slow_regularized,0.958728,0.947570,0.001819,0.011158
LightGBM_regularized,0.955490,0.945877,0.002011,0.009613


### LightGBM tuning insight

The deeper LightGBM configuration achieved the best mean validation AUC among the tested configurations.

Compared to the baseline LightGBM, it improves validation AUC, but also increases the train-validation gap.  
This means the model captures more signal, but it is also more flexible and has a higher risk of overfitting.

Therefore, the deeper LightGBM is currently the best candidate by validation AUC, while the baseline LightGBM remains a safer lower-variance alternative.  
In the next stage, we will compare these candidates using additional diagnostics such as ROC curves, fold stability, and possibly time-aware validation before deciding which model to use for the final submission.

## 6. Time-Aware Validation

The EDA showed that the test set comes from a later time period than most of the training set.  
Therefore, random Stratified K-Fold CV may be slightly optimistic.

To check future generalization, we create a time-aware validation split:
- train on earlier course dates,
- validate on the latest part of the training period.

This better simulates the hidden test setting, where the model predicts future registrations.

In [22]:
def evaluate_time_split(model, X_full, y, train_fe, split_quantile=0.8, model_name="model"):
    """
    Train on earlier course dates and validate on later course dates.
    This simulates future generalization.
    """
    dates = pd.to_datetime(train_fe["Course_Start_Date"], errors="coerce")
    split_date = dates.quantile(split_quantile)

    train_mask = dates < split_date
    valid_mask = dates >= split_date

    X_train = X_full.loc[train_mask].copy()
    X_valid = X_full.loc[valid_mask].copy()
    y_train = y.loc[train_mask].copy()
    y_valid = y.loc[valid_mask].copy()

    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    train_pred = pipeline.predict_proba(X_train)[:, 1]
    valid_pred = pipeline.predict_proba(X_valid)[:, 1]

    train_auc = roc_auc_score(y_train, train_pred)
    valid_auc = roc_auc_score(y_valid, valid_pred)

    print("Model:", model_name)
    print("Split date:", split_date)
    print("Train rows:", X_train.shape[0])
    print("Validation rows:", X_valid.shape[0])
    print("Train AUC:", round(train_auc, 5))
    print("Time validation AUC:", round(valid_auc, 5))
    print("Gap:", round(train_auc - valid_auc, 5))

    return {
        "model": model_name,
        "split_date": split_date,
        "train_rows": X_train.shape[0],
        "valid_rows": X_valid.shape[0],
        "train_auc": train_auc,
        "time_valid_auc": valid_auc,
        "gap": train_auc - valid_auc,
        "pipeline": pipeline,
    }

In [23]:
lgbm_baseline_model = LGBMClassifier(
    n_estimators=600,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective="binary",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

lgbm_deeper_model = LGBMClassifier(
    n_estimators=700,
    learning_rate=0.025,
    num_leaves=48,
    max_depth=-1,
    min_child_samples=40,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=0.8,
    objective="binary",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

time_baseline_result = evaluate_time_split(
    lgbm_baseline_model,
    X_model,
    y,
    train_fe,
    split_quantile=0.8,
    model_name="LightGBM_baseline_time_split"
)

time_deeper_result = evaluate_time_split(
    lgbm_deeper_model,
    X_model,
    y,
    train_fe,
    split_quantile=0.8,
    model_name="LightGBM_deeper_time_split"
)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Model: LightGBM_baseline_time_split
Split date: 2016-12-19 00:00:00
Train rows: 50735
Validation rows: 12729
Train AUC: 0.97009
Time validation AUC: 0.90203
Gap: 0.06806


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Model: LightGBM_deeper_time_split
Split date: 2016-12-19 00:00:00
Train rows: 50735
Validation rows: 12729
Train AUC: 0.97784
Time validation AUC: 0.90243
Gap: 0.07542


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [24]:
time_validation_summary = pd.DataFrame([
    {
        "model": time_baseline_result["model"],
        "train_auc": time_baseline_result["train_auc"],
        "time_valid_auc": time_baseline_result["time_valid_auc"],
        "gap": time_baseline_result["gap"],
    },
    {
        "model": time_deeper_result["model"],
        "train_auc": time_deeper_result["train_auc"],
        "time_valid_auc": time_deeper_result["time_valid_auc"],
        "gap": time_deeper_result["gap"],
    },
]).sort_values("time_valid_auc", ascending=False)

time_validation_summary

,model,train_auc,time_valid_auc,gap
1,LightGBM_deeper_time_split,0.977844,0.902426,0.075418
0,LightGBM_baseline_time_split,0.970090,0.902033,0.068056


### Time-aware validation insight

The time-aware validation AUC is much lower than the random cross-validation AUC.  
This suggests that the dataset contains temporal distribution shift: the model performs very well when validation rows are randomly sampled from the same period, but the task becomes harder when predicting later course registrations.

The deeper LightGBM configuration still performs slightly better than the baseline model on the time split, but the difference is very small.  
At the same time, the deeper model has a larger train-validation gap, which means it is more flexible and may be more sensitive to overfitting.

Therefore, we will keep both LightGBM configurations as candidates:
- `LightGBM_deeper` as the strongest random-CV model,
- `LightGBM_baseline` as a safer lower-variance alternative.

The final model choice should consider both cross-validation AUC and time-aware validation, not only leaderboard performance.